# Case Studies

## Case Study 1: Google Play Policy Compliance

We first analyse co-transmission of PII types with the Google
Advertising ID (GAID) across instrumentation approaches. Co-transmission
occurs when a data recipient transmits both the GAID and another PII
type, either within the same HTTP request (same-request co-transmission)
or across separate requests to the same DR. This analysis supports the
policy compliance discussion in Section 7.1

**Paper reference:** $\S$ 7.1, Table 7

In [5]:
# Setup and data loading
import pandas as pd
from analysis_utils.shared_setup import *

pii_df = pd.read_parquet("../data/pii_observed.parquet")
# Let us remove unknown related entries.
pii_df = pii_df[~pii_df["company"].str.contains("unknown", case=False)]
pii_df["valid"] = pii_df.apply(lambda r: r["app"] in r["app_test"], axis=1)
pii_df = pii_df[pii_df["valid"]].copy()

inst_approaches = sorted(pii_df["vantage_type"].unique())

mark_print(
    f"Loaded **{len(pii_df):,}** PII observations across "
    f"**{pii_df['company'].nunique()}** data recipients and "
    f"**{pii_df['pii_type'].nunique()}** PII types."
)

Loaded **76,779** PII observations across **184** data recipients and **16** PII types.

### Same-Request Co-Transmission Detection

Identify (DR, PII type, approach) tuples where a non-GAID PII type
co-occurs with the Advertising ID within the same HTTP request
(matched by UUID). Same-request co-transmission is a stronger signal
of linkage than cross-request co-transmission.

In [6]:
# --- Cell: Co-occurrence detection ---
def find_adid_cooccurrences(df):
    """Find (company, pii_type, vantage) where PII co-occurs with
    Advertising ID in the same request (matched by UUID)."""
    adid_requests = (
        df[df["pii_type"] == "Advertising ID"]
        [["company", "uuid", "vantage_type"]]
        .drop_duplicates()
    )
    other_pii = df[df["pii_type"] != "Advertising ID"]
    cooccur = other_pii.merge(
        adid_requests, on=["company", "uuid", "vantage_type"], how="inner"
    )
    uuid_cooccur = (
        cooccur[["company", "pii_type", "uuid", "vantage_type"]]
        .drop_duplicates()
    )
    cooccur_unique = (
        cooccur[["company", "pii_type", "vantage_type"]]
        .drop_duplicates()
    )
    return uuid_cooccur, cooccur_unique

uuid_cooccur, cooccur = find_adid_cooccurrences(pii_df)

same_request_counts = (
    uuid_cooccur
    .groupby(["pii_type", "vantage_type"])
    .agg(data_recipients=("company", "nunique"))
    .reset_index()
)

total_cotransmit = f"{cooccur['company'].nunique() / pii_df['company'].nunique() * 100:.1f}%"

mark_print(
    f"Detected **{len(cooccur)}** unique (DR, PII type, approach) "
    f"co-transmission tuples across **{cooccur['company'].nunique()}** "
    f"data recipients. Therefore, **{cooccur['company'].nunique()} out of "
    f"{pii_df['company'].nunique()} ({total_cotransmit})** may be violating "
    "Google Play Privacy Policy."
)

Detected **247** unique (DR, PII type, approach) co-transmission tuples across **60** data recipients. Therefore, **60 out of 184 (32.6%)** may be violating Google Play Privacy Policy.

### Policy Classification

Google Play policy prohibits co-transmission of the Advertising ID
with persistent device identifiers without explicit user consent.
We classify PII types into prohibited identifiers (no consent exception)
and policy-relevant identifiers (require explicit consent).

In [7]:
# Policy classification
import numpy as np
import re
from tabulate import tabulate

prohibited_types = {
    "Android ID", "App Set ID", "Firebase App Instance ID",
    "Firebase Installation ID", "GSF ID", "Device IMEI", "Device Serial",
}

def co_transmission_stats(df, approaches):
    """Per-approach co-transmission statistics for prohibited and
    policy-relevant PII types with GAID."""
    gaid_drs = set(df[df["pii_type"] == "Advertising ID"]["company"].unique())
    prohibited_drs = set(df[df["pii_type"].isin(prohibited_types)]["company"].unique())
    policy_types = set(df["pii_type"].unique()) - prohibited_types - {"Advertising ID"}
    policy_drs = set(df[df["pii_type"].isin(policy_types)]["company"].unique())

    return {
        "all_drs": set(df["company"].unique()),
        "gaid_drs": gaid_drs,
        "cotransmit_prohibited": gaid_drs & prohibited_drs,
        "cotransmit_policy": gaid_drs & policy_drs,
    }

vantage_results = {}
for vp in inst_approaches:
    vp_df = pii_df[pii_df["vantage_type"] == vp]
    vantage_results[vp] = co_transmission_stats(vp_df, inst_approaches)

# Summary table
ct_rows = []
for vp in inst_approaches:
    r = vantage_results[vp]
    ct_rows.append({
        "Approach": vp,
        "DRs with GAID": len(r["gaid_drs"]),
        "GAID + Prohibited": len(r["cotransmit_prohibited"]),
        "GAID + Policy-Relevant": len(r["cotransmit_policy"]),
    })

mark_print("**Co-transmission summary per approach:**")
mark_print(tabulate(ct_rows, headers="keys", tablefmt="pipe", showindex=False))

**Co-transmission summary per approach:**

| Approach   |   DRs with GAID |   GAID + Prohibited |   GAID + Policy-Relevant |
|:-----------|----------------:|--------------------:|-------------------------:|
| Advaita    |             116 |                  54 |                       26 |
| Aosp-Sci   |              61 |                  37 |                        4 |
| Frida      |              91 |                  44 |                        5 |
| Mitm       |              93 |                  48 |                       12 |

### Combined Co-Transmission Table (Table 7)

For each PII type, report the number of DRs that co-transmit that
type with the Advertising ID. Values show total DRs (same-request
co-transmissions in parentheses). The Advaita-only column counts DRs
whose co-transmission is invisible to all existing techniques.

In [8]:
# --- Cell: Build co-transmission table ---
# Per-PII type: DRs co-transmitting with GAID (across all requests)
ad_id_by_vp = {
    vp: set(pii_df[
        (pii_df["pii_type"] == "Advertising ID") & (pii_df["vantage_type"] == vp)
    ]["company"].unique())
    for vp in inst_approaches
}

co_all_rows = []
for pii in sorted(pii_df["pii_type"].unique()):
    if pii == "Advertising ID":
        continue
    by_vp = {
        vp: set(pii_df[
            (pii_df["pii_type"] == pii) & (pii_df["vantage_type"] == vp)
        ]["company"].unique())
        for vp in inst_approaches
    }
    co_by_vp = {vp: by_vp[vp] & ad_id_by_vp[vp] for vp in inst_approaches}
    all_co = set().union(*co_by_vp.values())
    advaita_co = co_by_vp.get("Advaita", set())
    others_co = set().union(*(co_by_vp[v] for v in inst_approaches if v != "Advaita"))
    advaita_only = advaita_co - others_co

    co_all_rows.append({
        "pii_type": pii,
        "total": len(all_co),
        **{f"{v}_count": len(co_by_vp[v]) for v in inst_approaches},
        "advaita_only": len(advaita_only),
        "advaita_only_DRs": sorted(advaita_only),
    })

co_all_df = pd.DataFrame(co_all_rows)

# Per-PII type: same-request co-transmission counts
def detection_stats(df, group_col="pii_type"):
    rows = []
    for val in sorted(df[group_col].unique()):
        sub = df[df[group_col] == val]
        by_vp = {
            vp: set(sub[sub["vantage_type"] == vp]["company"])
            for vp in inst_approaches
        }
        advaita = by_vp.get("Advaita", set())
        others = set().union(*(by_vp[v] for v in inst_approaches if v != "Advaita"))
        advaita_only = advaita - others
        rows.append({
            group_col: val,
            "total": len(set().union(*by_vp.values())),
            **{f"{v}_count": len(by_vp.get(v, set())) for v in inst_approaches},
            "advaita_only": len(advaita_only),
            "advaita_only_DRs": sorted(advaita_only),
        })
    return pd.DataFrame(rows)

stats_co = detection_stats(cooccur)

# Merge: all co-transmissions with same-request co-transmissions
merged = co_all_df.merge(
    stats_co, on="pii_type", how="left", suffixes=("_all", "_co")
).fillna(0)

for col in ["advaita_only_DRs_all", "advaita_only_DRs_co"]:
    merged[col] = merged[col].apply(lambda x: x if isinstance(x, list) else [])

# Format combined table: "total (same-request)"
fmt_rows = []
from collections import Counter

totals, co_total = Counter(), Counter()
for _, r in merged.iterrows():
    row = {"PII Type": r["pii_type"]}
    row["Combined"] = f"{int(r['total_all'])} ({int(r['total_co'])})"
    totals["Combined"] += int(r['total_all'])
    co_total["Combined"] += int(r['total_co'])
    for vp in inst_approaches:
        total = int(r[f"{vp}_count_all"])
        co = int(r[f"{vp}_count_co"])
        row[vp] = f"{total} ({co})"
        totals[vp] += total
        co_total[vp] += co
    ao_all = int(r["advaita_only_all"]) if isinstance(r["advaita_only_all"], (int, float)) else 0
    ao_co = int(r["advaita_only_co"]) if isinstance(r["advaita_only_co"], (int, float)) else 0
    row["Advaita-only"] = f"{ao_all} ({ao_co})"
    totals["Advaita-only"] += ao_all
    co_total["Advaita-only"] += ao_co
    row["_drs_all"] = r["advaita_only_DRs_all"]
    row["_drs_co"] = r["advaita_only_DRs_co"]
    fmt_rows.append(row)

total_row = {"PII Type": "Total (PII-DR Transmissions)"}
total_row["Combined"] = f"{totals['Combined']} ({co_total['Combined']})"
for vp in inst_approaches:
    total_row[vp] = f"{totals[vp]} ({co_total[vp]})"
total_row["Advaita-only"] = f"{totals['Advaita-only']} ({co_total['Advaita-only']})"
total_row["_drs_all"] = []
total_row["_drs_co"] = []

combined_table = pd.DataFrame(fmt_rows).sort_values("PII Type").reset_index(drop=True)
combined_table = pd.concat([combined_table, pd.DataFrame([total_row])], ignore_index=True)

display_cols = [c for c in combined_table.columns if not c.startswith("_")]
mark_print("**Table 7: Co-transmission of PII with Advertising ID.** "
           "Values show DR count (same-request co-transmissions). "
           "Advaita-only counts DRs invisible to all existing techniques.")
mark_print(tabulate(
    combined_table[display_cols], headers="keys", tablefmt="pipe", showindex=False
))

**Table 7: Co-transmission of PII with Advertising ID.** Values show DR count (same-request co-transmissions). Advaita-only counts DRs invisible to all existing techniques.

| PII Type                     | Combined   | Advaita   | Aosp-Sci   | Frida   | Mitm    | Advaita-only   |
|:-----------------------------|:-----------|:----------|:-----------|:--------|:--------|:---------------|
| Android ID                   | 33 (28)    | 27 (25)   | 16 (15)    | 25 (22) | 26 (21) | 4 (3)          |
| App Set ID                   | 31 (30)    | 29 (28)   | 21 (20)    | 25 (24) | 25 (25) | 2 (2)          |
| Contact Email                | 2 (0)      | 2 (0)     | 0 (0)      | 0 (0)   | 0 (0)   | 2 (0)          |
| Contact Phone                | 8 (2)      | 7 (2)     | 1 (0)      | 2 (0)   | 2 (1)   | 5 (1)          |
| Device IMEI                  | 0 (0)      | 0 (0)     | 0 (0)      | 0 (0)   | 0 (0)   | 0 (0)          |
| Firebase App Instance ID     | 3 (1)      | 2 (0)     | 2 (1)      | 2 (0)   | 2 (0)   | 0 (0)          |
| Firebase Installation ID     | 18 (8)     | 13 (6)    | 10 (6)     | 12 (7)  | 16 (8)  | 1 (0)          |
| GSF ID                       | 2 (1)      | 2 (1)     | 0 (0)      | 0 (0)   | 2 (1)   | 0 (0)          |
| Gmail Profile Picture        | 8 (1)      | 5 (1)     | 1 (0)      | 1 (1)   | 4 (0)   | 2 (0)          |
| Google Account Email         | 17 (7)     | 14 (5)    | 2 (1)      | 4 (2)   | 5 (1)   | 7 (3)          |
| Google Account Name          | 14 (5)     | 11 (3)    | 2 (1)      | 3 (2)   | 6 (2)   | 3 (0)          |
| Location                     | 12 (9)     | 11 (8)    | 1 (0)      | 1 (1)   | 5 (4)   | 6 (4)          |
| Photos Location              | 1 (0)      | 1 (0)     | 0 (0)      | 0 (0)   | 0 (0)   | 1 (0)          |
| Router Location              | 2 (1)      | 2 (1)     | 1 (1)      | 0 (0)   | 0 (0)   | 1 (0)          |
| Video Location               | 1 (0)      | 1 (0)     | 0 (0)      | 0 (0)   | 0 (0)   | 1 (0)          |
| Total (PII-DR Transmissions) | 152 (93)   | 127 (80)  | 57 (45)    | 75 (59) | 93 (63) | 35 (13)        |

### Advaita-Exclusive DR Details (Table 7)

List the specific data recipients whose co-transmission with the
Advertising ID is detected exclusively by Advaita, per PII type.

In [6]:
# --- Cell: Advaita-only DR details ---
detail_rows = []
for _, r in combined_table.iterrows():
    drs = r["_drs_all"]
    if drs:
        drs_co = set(r["_drs_co"]) if isinstance(r["_drs_co"], list) else set()
        detail_rows.append({
            "PII Type": r["PII Type"],
            "Exclusive DRs": len(drs),
            "Data Recipients": ", ".join(sorted(drs)),
        })

if detail_rows:
    mark_print("**Table 7: Advaita-exclusive co-transmitting DRs by PII type:**")
    mark_print(tabulate(detail_rows, headers="keys", tablefmt="pipe", showindex=False))

**Table 7: Advaita-exclusive co-transmitting DRs by PII type:**

| PII Type                 |   Exclusive DRs | Data Recipients                                                       |
|:-------------------------|----------------:|:----------------------------------------------------------------------|
| Android ID               |               4 | Adobe, MobileFuse, Snapchat, The Trade Desk                           |
| App Set ID               |               2 | Funsdata-JS, Pixalate                                                 |
| Contact Email            |               2 | ByteDance, Snapchat                                                   |
| Contact Phone            |               5 | ByteDance, CBS/Paramount, Google, Mixpanel, Snapchat                  |
| Firebase Installation ID |               1 | Snapchat                                                              |
| Gmail Profile Picture    |               2 | ByteDance, CBS/Paramount                                              |
| Google Account Email     |               7 | Adobe, Amplitude, ByteDance, CBS/Paramount, FreeWheel, Meta, Singular |
| Google Account Name      |               3 | ByteDance, CBS/Paramount, Snapchat                                    |
| Location                 |               6 | CBS/Paramount, DoubleVerify, FreeWheel, Mixpanel, Tapjoy, Verve       |
| Photos Location          |               1 | Meta                                                                  |
| Router Location          |               1 | Meta                                                                  |
| Video Location           |               1 | Meta                                                                  |

### Summary Statistics  (Table 7)

Aggregate co-transmission counts and identify notable patterns:
dagger-marked PII types (≥40% Advaita-exclusive), fully exclusive
types (zero detection by all existing techniques), and the largest
observability gap.

In [9]:
# --- Cell: Summary statistics ---
def parse_cell(val):
    m = re.match(r"(\d+)\s*\((\d+)\)", str(val))
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

# Column totals
total_cols = ["Advaita", "Frida", "Mitm", "Aosp-Sci", "Advaita-only", "Combined"]
totals = {}
for col in total_cols:
    if col in combined_table.columns:
        t_all, t_co = 0, 0
        for val in combined_table[col]:
            a, c = parse_cell(val)
            t_all += a
            t_co += c
        totals[col] = (t_all, t_co)

total_rows = [{"Column": col, "DR-PII Pairs": f"{a} ({c})"}
              for col, (a, c) in totals.items()]
mark_print("**Column totals:**")
mark_print(tabulate(total_rows, headers="keys", tablefmt="pipe", showindex=False))

# Dagger: >=40% exclusive to Advaita
dagger_types = []
for _, r in combined_table.iterrows():
    adv_all, _ = parse_cell(r["Advaita"])
    only_all, _ = parse_cell(r["Advaita-only"])
    if adv_all > 0 and (only_all / adv_all) >= 0.40:
        dagger_types.append(r["PII Type"])

# Fully exclusive (zero across all existing techniques)
fully_exclusive = []
for _, r in combined_table.iterrows():
    frida_all, _ = parse_cell(r["Frida"])
    mitm_all, _ = parse_cell(r["Mitm"])
    aosp_all, _ = parse_cell(r["Aosp-Sci"])
    if frida_all == 0 and mitm_all == 0 and aosp_all == 0:
        fully_exclusive.append(r["PII Type"])

# Largest gap
max_gap_type, max_gap_val, max_gap_adv = None, 0, 0
for _, r in combined_table.iterrows():
    adv_all, _ = parse_cell(r["Advaita"])
    only_all, _ = parse_cell(r["Advaita-only"])
    if only_all > max_gap_val:
        max_gap_val = only_all
        max_gap_adv = adv_all
        max_gap_type = r["PII Type"]

# Per-technique coverage relative to Advaita
adv_total = totals.get("Advaita", (0, 0))[0]

# Same-request prevalence
comb_total = totals.get("Combined", (0, 0))

mark_print(
    f"The table reports co-transmission across **{len(combined_table)}** PII types.\n\n"

    f"PII Types where ≥40% Advaita-exclusive: "
    f"{', '.join(dagger_types) if dagger_types else 'none'}.\n\n"

    f"**Fully exclusive types** (zero detection by existing techniques): "
    f"{', '.join(fully_exclusive) if fully_exclusive else 'none'}.\n\n"

    f"**Largest gap:** {max_gap_type} — {max_gap_val}/{max_gap_adv} exclusive "
    f"({max_gap_val / max_gap_adv * 100:.1f}% of Advaita detections).\n\n"

    + "".join(
        f"**{vp}:** {totals[vp][0]}/{adv_total} DR-PII pairs "
        f"({totals[vp][0] / adv_total * 100:.1f}% of Advaita). "
        for vp in ["Frida", "Mitm", "Aosp-Sci"] if vp in totals and adv_total > 0
    ) + "\n\n"

    f"**Same-request co-transmission prevalence:** {comb_total[1]}/{comb_total[0]} "
    f"({comb_total[1] / comb_total[0] * 100:.1f}% of all co-transmissions)."
)

**Column totals:**

| Column       | DR-PII Pairs   |
|:-------------|:---------------|
| Advaita      | 254 (160)      |
| Frida        | 150 (118)      |
| Mitm         | 186 (126)      |
| Aosp-Sci     | 114 (90)       |
| Advaita-only | 70 (26)        |
| Combined     | 304 (186)      |

The table reports co-transmission across **16** PII types.

PII Types where ≥40% Advaita-exclusive: Contact Email, Contact Phone, Gmail Profile Picture, Google Account Email, Location, Photos Location, Router Location, Video Location.

**Fully exclusive types** (zero detection by existing techniques): Contact Email, Device IMEI, Photos Location, Video Location.

**Largest gap:** Total (PII-DR Transmissions) — 35/127 exclusive (27.6% of Advaita detections).

**Frida:** 150/254 DR-PII pairs (59.1% of Advaita). **Mitm:** 186/254 DR-PII pairs (73.2% of Advaita). **Aosp-Sci:** 114/254 DR-PII pairs (44.9% of Advaita). 

**Same-request co-transmission prevalence:** 186/304 (61.2% of all co-transmissions).

### Persistent Device Identifier Transmissions

We examine transmissions of Device Serial, Device IMEI, and GSF ID.
These are persistent device identifiers that cannot be reset by
the user and are subject to the strictest policy restrictions under
Google Play guidelines.


In [10]:
# --- Cell: Persistent identifier analysis ---
all_observed = pd.read_parquet("../data/pii_observed_all.parquet")
pii_df = all_observed.copy()
pii_df["valid"] = pii_df.apply(lambda r: r["app"] in r["app_test"], axis=1)
pii_df = pii_df[pii_df["valid"]].copy()
pii_df = pii_df.drop(columns=[
    "ip", "port", "decoders", "pii_label", "http_stream_type",
    "app_test", "clientrandom", "vantage_type",
])
pii_df["app"] = pii_df["app"].str.split("$").str[0]

total_google_apps = pii_df[
    pii_df["app"].str.contains("com.google|com.android", regex=True)
]["app"].unique()

def get_persistent_df(df):
    df = map_libs(df)
    df["tls_lib_label"] = df["tls_lib_label"].fillna("n/a")
    df = df[df["pii_type"].str.contains("Device|GSF", regex=True, case=False, na=False)]
    df = (
        df.groupby(["pii_type", "app", "tls_lib_label"])
        .agg(
            data_recipients=("company", "nunique"),
            dr_names=("company", lambda x: ", ".join(sorted(x.dropna().unique()))),
        )
        .reset_index()
    )
    return df

persistent_df = get_persistent_df(pii_df.copy())
persistent_df = persistent_df[persistent_df["app"] != "None"]

google_apps = persistent_df[
    persistent_df["app"].str.contains("google|android", regex=True)
]["app"].unique()

mark_print(
    f"**{len(persistent_df['app'].unique())}** apps transmit persistent device "
    f"identifiers. Of these, **{len(google_apps)}** are Google/Android system apps "
    f"(out of **{len(total_google_apps)}** total Google apps in the dataset)."
)

# Per-app breakdown
insights = (
    persistent_df.groupby(["pii_type", "app"])
    .agg(
        tls_libraries=("tls_lib_label", lambda x: ", ".join(sorted(x.dropna().unique()))),
        data_recipients=("data_recipients", "sum"),
        dr_names=("dr_names", lambda x: ", ".join(sorted(set(
            dr.strip() for names in x.dropna() for dr in names.split(", ")
        )))),
    )
    .reset_index()
    .rename(columns={"dr_names": "Data Recipients"})
    .sort_values(["pii_type", "data_recipients"], ascending=[True, False])
)

mark_print("**Persistent identifier transmissions by app and PII type:**")
mark_print(tabulate(insights, headers="keys", tablefmt="pipe", showindex=False))

**3** apps transmit persistent device identifiers. Of these, **2** are Google/Android system apps (out of **2** total Google apps in the dataset).

**Persistent identifier transmissions by app and PII type:**

| pii_type    | app                             | tls_libraries              |   data_recipients | Data Recipients   |
|:------------|:--------------------------------|:---------------------------|------------------:|:------------------|
| Device IMEI | com.handcent.app.nextsms        | Android System (Conscrypt) |                 1 | Handcent Next SMS |
| GSF ID      | com.google.android.youtube      | YouTube (Cronet), n/a      |                 3 | Google, YouTube   |
| GSF ID      | com.google.android.apps.adwords | GMS (Conscrypt), n/a       |                 2 | Google            |

## Case Study 2 : Background Transmissions

This section examines PII transmitted by apps that were *not* the
foreground test target at the time of capture. These background
transmissions reveal data collection activity that occurs without
direct user interaction, often through system services or SDKs
running in other processes.

**Paper reference:** $\S$ 7.2


In [14]:
# --- Cell: Background transmission overview ---
pii_bg_orig = pd.read_parquet("../data/pii_observed_all.parquet")
print(f"pii_observed_all.parquet DF cols: {pii_bg_orig.columns} | {pii_bg_orig.size}")
pii_bg = pii_bg_orig.copy()
pii_bg = pii_bg[pii_bg["vantage_type"] == "Advaita"]
pii_bg = pii_bg[pii_bg["app"] != "None"]
pii_bg["app"] = pii_bg["app"].str.split("$").str[0]

total_google_apps_bg = pii_bg[
    pii_bg["app"].str.contains("com.google|com.android", regex=True)
]["app"].unique()

print(f"Unique DRs (foreground+background): {pii_bg['company'].nunique()}")
pii_bg["background"] = pii_bg.apply(lambda r: r["app"] not in r["app_test"], axis=1)
pii_bg = pii_bg[pii_bg["background"]].copy()
print(f"Unique DRs (background): {pii_bg['company'].nunique()}")

back_google_apps = pii_bg[
    pii_bg["app"].str.contains("com.google|com.android", regex=True)
]["app"].unique()

pii_bg = pii_bg.drop(columns=[
    "ip", "port", "decoders", "pii_label", "http_stream_type",
    "app_test", "clientrandom",
])
pii_bg = map_libs(pii_bg)
pii_bg["tls_lib_label"] = pii_bg["tls_lib_label"].fillna("n/a")
pii_bg = pii_bg[["pii_type", "vantage_type", "app", "tls_lib_label", "company"]]
n_apps = pii_bg["app"].nunique()
n_pii_types = pii_bg["pii_type"].nunique()
n_drs = pii_bg["company"].nunique()
n_observations = len(pii_bg)
dr_names = ", ".join(sorted(pii_bg["company"].unique()))

mark_print(
    f"**Background PII transmissions:** {n_observations:,} observations from "
    f"**{n_apps}** apps transmitting **{n_pii_types}** PII types to "
    f"**{n_drs}** data recipients ({dr_names}). "
    f"Of **{len(total_google_apps_bg)}** Google/Android apps in the dataset, "
    f"**{len(back_google_apps)}** transmit PII in the background."
)

pii_observed_all.parquet DF cols: Index(['app', 'host', 'ip', 'port', 'pii_type', 'time', 'dir', 'uuid', 'decoders', 'pii_label', 'http_stream_type', 'vantage_type', 'app_test', 'clientrandom', 'tls_lib', 'company'], dtype='str') | 1490672
Unique DRs (foreground+background): 154
Unique DRs (background): 16


**Background PII transmissions:** 3,981 observations from **18** apps transmitting **17** PII types to **16** data recipients (Among Us Chat, Android Services, AppsFlyer, ByteDance, Firebase, Google, Google AdManager, Google Maps, Google Play Store, Meta, SRM Data, Shein, SpeedChecker, Spotify, Temu, YouTube). Of **11** Google/Android apps in the dataset, **10** transmit PII in the background.

In [10]:
# --- Cell: Background transmission Google Apps ---
back_google_apps = pii_bg[
    pii_bg["app"].str.contains("com.google|com.android", regex=True)
]["app"].unique()

background_df = (
    pii_bg.groupby(["pii_type", "vantage_type", "app", "tls_lib_label"])
    .agg(data_recipients=("company", "nunique"))
    .reset_index()
)
background_df = background_df[background_df["app"] != "None"]

mark_print(
    f"**{len(total_google_apps_bg)}** Google/Android apps observed in total. "
    f"**{len(back_google_apps)}** transmitted PII in the background."
)
background_df

**11** Google/Android apps observed in total. **10** transmitted PII in the background.

,pii_type,vantage_type,app,tls_lib_label,data_recipients
0,Advertising ID,Advaita,com.android.vending,Android System (Conscrypt),1
1,Advertising ID,Advaita,com.android.vending,Google Play (Cronet),1
2,Advertising ID,Advaita,com.facebook.katana,Android System (Conscrypt),1
3,Advertising ID,Advaita,com.google.android.adservices.api,Android System (Conscrypt),2
4,Advertising ID,Advaita,com.google.android.apps.maps,GMS (Cronet),1
...,...,...,...,...,...
73,Location,Advaita,com.google.android.gsf,GMS (Cronet),2
74,Photos Copyright,Advaita,com.google.android.googlequicksearchbox,GMS (Conscrypt),1
75,Router Location,Advaita,com.appmind.radios.gt,Android System (Conscrypt),1
76,Router Location,Advaita,com.google.android.gsf,GMS (Conscrypt),1


In [11]:
# --- Cell: Background apps by PII breadth ---
app_pii_counts = (
    background_df.groupby("app")["pii_type"]
    .nunique()
    .sort_values(ascending=False)
)

bg_rows = []
transmission = 0
for app in app_pii_counts.index:
    types = sorted(background_df[background_df["app"] == app]["pii_type"].unique())
    bg_rows.append({
        "App": app,
        "PII Types": len(types),
        "Types Transmitted": ", ".join(types),
    })
    transmission += len(types)

mark_print(f"**Background apps ranked by number of distinct PII types transmitted (Total: {transmission}):**")
mark_print(tabulate(bg_rows, headers="keys", tablefmt="pipe", showindex=False))

**Background apps ranked by number of distinct PII types transmitted (Total: 65):**

| App                                     |   PII Types | Types Transmitted                                                                                                                                                                                                                                                          |
|:----------------------------------------|------------:|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| com.google.android.gsf                  |          16 | Advertising ID, Android ID, App Set ID, Contact Address, Contact Email, Contact Phone, Device IMEI, Device Serial, Firebase App Instance ID, Firebase Installation ID, GSF ID, Gmail Profile Picture, Google Account Email, Google Account Name, Location, Router Location |
| com.google.android.googlequicksearchbox |           8 | Advertising ID, Contact Address, Device Serial, Firebase Installation ID, Gmail Profile Picture, Google Account Email, Google Account Name, Photos Copyright                                                                                                               |
| com.google.android.apps.maps            |           7 | Advertising ID, Contact Phone, Firebase Installation ID, Gmail Profile Picture, Google Account Email, Google Account Name, Location                                                                                                                                        |
| com.android.vending                     |           6 | Advertising ID, Device Serial, GSF ID, Gmail Profile Picture, Google Account Email, Google Account Name                                                                                                                                                                    |
| com.google.android.gm                   |           6 | Advertising ID, Firebase Installation ID, GSF ID, Gmail Profile Picture, Google Account Email, Google Account Name                                                                                                                                                         |
| com.zzkko                               |           4 | Advertising ID, Android ID, Firebase Installation ID, Google Account Email                                                                                                                                                                                                 |
| com.google.android.youtube              |           3 | Advertising ID, GSF ID, Google Account Name                                                                                                                                                                                                                                |
| com.appmind.radios.gt                   |           2 | Location, Router Location                                                                                                                                                                                                                                                  |
| com.einnovation.temu                    |           2 | Firebase Installation ID, Google Account Name                                                                                                                                                                                                                              |
| com.instagram.android                   |           2 | Advertising ID, Router Location                                                                                                                                                                                                                                            |
| walkie.talkie.among.us.friends          |           2 | Firebase Installation ID, Google Account Name                                                                                                                                                                                                                              |
| com.google.android.adservices.api       |           1 | Advertising ID                                                                                                                                                                                                                                                             |
| com.google.android.apps.photos          |           1 | Google Account Name                                                                                                                                                                                                                                                        |
| com.google.android.apps.pixel.dcservice |           1 | GSF ID                                                                                                                                                                                                                                                                     |
| com.google.android.calendar             |           1 | Google Account Email                                                                                                                                                                                                                                                       |
| com.facebook.katana                     |           1 | Advertising ID                                                                                                                                                                                                                                                             |
| com.spotify.music                       |           1 | Advertising ID                                                                                                                                                                                                                                                             |
| com.zhiliaoapp.musically                |           1 | Advertising ID                                                                                                                                                                                                                                                             |

In [12]:
# --- Cell: Background transmissions by TLS library ---
lib_totals = (
    background_df.groupby("tls_lib_label")["data_recipients"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"tls_lib_label": "TLS Library", "data_recipients": "Total DR Transmissions"})
)

mark_print("**Background transmission volume by TLS library:**")
mark_print(tabulate(lib_totals, headers="keys", tablefmt="pipe", showindex=False))

**Background transmission volume by TLS library:**

| TLS Library                 |   Total DR Transmissions |
|:----------------------------|-------------------------:|
| GMS (Cronet)                |                       31 |
| GMS (Conscrypt)             |                       24 |
| Android System (Conscrypt)  |                       20 |
| Google Play (Cronet)        |                        8 |
| Google QuickSearch (Cronet) |                        4 |
| YouTube (Cronet)            |                        3 |
| Meta (Instagram - Fizz)     |                        2 |
| TikTok (BoringSSL)          |                        1 |

In [13]:
pii_bg

,pii_type,vantage_type,app,tls_lib_label,company
58031,Advertising ID,Advaita,com.android.vending,Android System (Conscrypt),Google Play Store
58042,Firebase Installation ID,Advaita,com.google.android.gsf,GMS (Cronet),Firebase
58043,Advertising ID,Advaita,com.google.android.gsf,GMS (Cronet),Firebase
58044,GSF ID,Advaita,com.google.android.apps.pixel.dcservice,GMS (Cronet),Google
58045,GSF ID,Advaita,com.google.android.apps.pixel.dcservice,GMS (Cronet),Google
...,...,...,...,...,...
93162,GSF ID,Advaita,com.google.android.gsf,GMS (Cronet),Google
93163,GSF ID,Advaita,com.google.android.gsf,GMS (Cronet),Google
93164,GSF ID,Advaita,com.google.android.gsf,GMS (Cronet),Google
93165,GSF ID,Advaita,com.google.android.gsf,GMS (Cronet),Google
